# Ollama (2026 업데이트판)

Ollama는 로컬 환경에서 LLM과 임베딩 모델을 간단한 명령어로 내려받아 실행할 수 있게 해주는 오픈소스 도구입니다.

- [공식 웹사이트/설치](https://ollama.com/)

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| 임포트 | `langchain_community.embeddings.OllamaEmbeddings` | `langchain_ollama.OllamaEmbeddings` (전용 패키지) |
| 기본 모델 | `nomic-embed-text` | 한국어 문장에는 다국어 모델(`bge-m3` 등) 권장 |
| 유사도 | 내적 | 정규화된 코사인 유사도 헬퍼 |

`langchain_community`의 `OllamaEmbeddings`는 오래전에 deprecated 되었고, `langchain-community` 패키지 자체도 2026년 5월 지원 종료되었습니다. Ollama 연동은 Ollama 전용 패키지 `langchain-ollama`를 사용합니다.

In [ ]:
%pip install -qU langchain-ollama python-dotenv numpy

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv`로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()`는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.
- 참고: 임베딩 호출(`embed_query`, `embed_documents`)은 Runnable이 아니어서 LangSmith에 트레이스가 남지 않습니다. 이 챕터에서는 없어도 되는 설정이지만, 이후 체인/에이전트 실습과 형태를 맞추기 위해 둡니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH08-Embeddings")

In [ ]:
texts = [
    "안녕, 만나서 반가워.",
    "LangChain simplifies the process of building applications with large language models",
    "랭체인 한국어 튜토리얼은 LangChain의 공식 문서, cookbook 및 다양한 실용 예제를 바탕으로 하여 사용자가 LangChain을 더 쉽고 효과적으로 활용할 수 있도록 구성되어 있습니다. ",
    "LangChain은 초거대 언어모델로 애플리케이션을 구축하는 과정을 단순화합니다.",
    "Retrieval-Augmented Generation (RAG) is an effective technique for improving AI responses.",
]

**지원 임베딩 모델 확인**: https://ollama.com/search?c=embedding

터미널에서 모델을 먼저 내려받습니다.

```bash
ollama pull bge-m3            # 다국어(한국어 포함), 1024차원
ollama pull nomic-embed-text  # 영어 중심, 가벼움
```

**모델 선택 참고**
- `nomic-embed-text`는 영어 중심 모델이라 한국어 문장 비교에 불리합니다. 또 모델 카드에 따르면 쿼리에는 `search_query: `, 문서에는 `search_document: ` 접두어를 붙여야 제 성능이 나옵니다.
- `bge-m3`는 원본 노트북에서 주석으로 언급한 모델로, 이제 Ollama 공식 라이브러리에 올라와 있어 `chatfire/bge-m3:q8_0` 같은 커뮤니티 태그 대신 `bge-m3`로 바로 받을 수 있습니다. 접두어가 필요 없습니다.

In [ ]:
from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    # model="nomic-embed-text",
    # base_url="http://localhost:11434",  # 원격 Ollama 서버를 쓸 때 지정
)

`Query`를 임베딩합니다.

In [ ]:
query = "LangChain 에 대해서 상세히 알려주세요."

embedded_query = ollama_embeddings.embed_query(query)
len(embedded_query)  # 임베딩 차원

문서를 임베딩합니다.

In [ ]:
embedded_documents = ollama_embeddings.embed_documents(texts)

유사도 계산 결과를 출력합니다.

**변경점**: 원본은 `query @ documents.T`(내적)만 사용했습니다. 내적은 벡터가 **정규화되어 있을 때만** 코사인 유사도와 같습니다. 모델마다 정규화 여부가 다르므로, 아래처럼 명시적으로 정규화한 뒤 코사인 유사도를 계산하는 헬퍼를 쓰는 편이 안전합니다.

In [ ]:
import numpy as np


def cosine_scores(query_vec, doc_vecs):
    q = np.asarray(query_vec, dtype=np.float32)
    d = np.asarray(doc_vecs, dtype=np.float32)
    q = q / np.linalg.norm(q)
    d = d / np.linalg.norm(d, axis=1, keepdims=True)
    return d @ q


def print_ranking(query, query_vec, doc_vecs, docs):
    scores = cosine_scores(query_vec, doc_vecs)
    print(f"[Query] {query}\n" + "=" * 40)
    for rank, idx in enumerate(scores.argsort()[::-1]):
        print(f"[{rank}] 유사도: {scores[idx]:.3f} | {docs[idx]}\n")

In [ ]:
print_ranking(query, embedded_query, embedded_documents, texts)